<a href="https://colab.research.google.com/github/shivamkumar516/EntityGuard/blob/main/project_khushahal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import csv
import sqlite3
from pathlib import Path

# Project paths
# Since __file__ is not available in Jupyter/Colab, we use Path.cwd()
ROOT = Path.cwd().resolve()

DB_PATH = ROOT / "data" / "processed" / "entityguard.db"

# Fixed: Correctly prepend ROOT and assign correct filenames
DBLP_FILE = ROOT / "/content/drive/MyDrive/DBLP.csv"
SCHOLAR_FILE = ROOT / "/content/drive/MyDrive/Scholar.csv"
GOLD_FILE = ROOT / "/content/drive/MyDrive/DBLP-Scholar_perfectMapping.csv"


def parse_year(value):
    """Convert a year into an integer, or None if it is missing."""
    value = (value or "").strip()

    if not value:
        return None

    try:
        return int(value)
    except ValueError:
        return None


def create_tables(connection):
    """Create the three tables required for dataset importing."""

    connection.executescript("""
        CREATE TABLE IF NOT EXISTS DBLP_RECORD (
            record_id TEXT PRIMARY KEY,
            title TEXT,
            authors TEXT,
            venue TEXT,
            year INTEGER
        );

        CREATE TABLE IF NOT EXISTS SCHOLAR_RECORD (
            record_id TEXT PRIMARY KEY,
            title TEXT,
            authors TEXT,
            venue TEXT,
            year INTEGER
        );

        CREATE TABLE IF NOT EXISTS GOLD_MATCH (
            dblp_id TEXT NOT NULL,
            scholar_id TEXT NOT NULL,
            PRIMARY KEY (dblp_id, scholar_id)
        );
    """)


def import_publications(connection, file_path, table_name):
    """Import DBLP or Scholar publication records."""

    query = f"""
        INSERT OR REPLACE INTO {table_name}
        (record_id, title, authors, venue, year)
        VALUES (?, ?, ?, ?, ?)
    """

    count = 0

    with open(file_path, "r", encoding="utf-8-sig",
              errors="replace", newline="") as file:

        reader = csv.DictReader(file)

        for row in reader:
            record_id = (row.get("id") or "").strip()

            if not record_id:
                continue

            values = (
                record_id,
                (row.get("title") or "").strip(),
                (row.get("authors") or "").strip(),
                (row.get("venue") or "").strip(),
                parse_year(row.get("year"))
            )

            connection.execute(query, values)
            count += 1

    return count


def import_gold_matches(connection, file_path):
    """Import verified DBLP-Scholar matching pairs."""

    query = """
        INSERT OR REPLACE INTO GOLD_MATCH
        (dblp_id, scholar_id)
        VALUES (?, ?)
    """

    count = 0

    with open(file_path, "r", encoding="utf-8-sig",
              errors="replace", newline="") as file:

        reader = csv.DictReader(file)

        for row in reader:
            dblp_id = (row.get("idDBLP") or "").strip()
            scholar_id = (row.get("idScholar") or "").strip()

            if not dblp_id or not scholar_id:
                continue

            connection.execute(query, (dblp_id, scholar_id))
            count += 1

    return count


def main():
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)

    # Check that all input files exist
    for path in [DBLP_FILE, SCHOLAR_FILE, GOLD_FILE]:
        if not path.exists():
            raise FileNotFoundError(f"Dataset file not found: {path}")

    connection = sqlite3.connect(DB_PATH)

    try:
        create_tables(connection)

        dblp_count = import_publications(
            connection, DBLP_FILE, "DBLP_RECORD"
        )

        scholar_count = import_publications(
            connection, SCHOLAR_FILE, "SCHOLAR_RECORD"
        )

        gold_count = import_gold_matches(
            connection, GOLD_FILE
        )

        connection.commit()

        print("\n===== ENTITYGUARD DATA IMPORT =====")
        print("DBLP records imported:", dblp_count)
        print("Scholar records imported:", scholar_count)
        print("Gold matches imported:", gold_count)

        print("\n===== DATABASE VERIFICATION =====")

        for table in ["DBLP_RECORD", "SCHOLAR_RECORD", "GOLD_MATCH"]:
            result = connection.execute(
                f"SELECT COUNT(*) FROM {table}"
            ).fetchone()

            print(f"{table}: {result[0]} records")

        missing_years = connection.execute(
            "SELECT COUNT(*) FROM SCHOLAR_RECORD WHERE year IS NULL"
        ).fetchone()[0]

        print("Scholar records with missing year:", missing_years)
        print("\nDatabase location:", DB_PATH)
        print("DATA IMPORT COMPLETED SUCCESSFULLY!")

    except Exception:
        connection.rollback()
        raise

    finally:
        connection.close()


if __name__ == "__main__":
    main()


===== ENTITYGUARD DATA IMPORT =====
DBLP records imported: 2616
Scholar records imported: 64263
Gold matches imported: 5347

===== DATABASE VERIFICATION =====
DBLP_RECORD: 2616 records
SCHOLAR_RECORD: 64263 records
GOLD_MATCH: 5347 records
Scholar records with missing year: 34790

Database location: /content/data/processed/entityguard.db
DATA IMPORT COMPLETED SUCCESSFULLY!


DATA IMPORT COMPLETED SUCCESSFULLY!

In [ ]:
import sqlite3
from pathlib import Path

# Project path
# Fixed: Jupyter/Colab environments do not have __file__, use Path.cwd() instead
ROOT = Path.cwd().resolve()

DB_PATH = ROOT / "data" / "processed" / "entityguard.db"


def create_indexes(connection):
    """
    Create B-tree indexes to improve database search performance.
    """

    indexes = [
        # DBLP indexes
        """
        CREATE INDEX IF NOT EXISTS idx_dblp_year
        ON DBLP_RECORD(year)
        """,

        """
        CREATE INDEX IF NOT EXISTS idx_dblp_title
        ON DBLP_RECORD(title)
        """,

        """
        CREATE INDEX IF NOT EXISTS idx_dblp_venue
        ON DBLP_RECORD(venue)
        """,

        # Scholar indexes
        """
        CREATE INDEX IF NOT EXISTS idx_scholar_year
        ON SCHOLAR_RECORD(year)
        """,

        """
        CREATE INDEX IF NOT EXISTS idx_scholar_title
        ON SCHOLAR_RECORD(title)
        """,

        """
        CREATE INDEX IF NOT EXISTS idx_scholar_venue
        ON SCHOLAR_RECORD(venue)
        """,

        # Gold matching indexes
        """
        CREATE INDEX IF NOT EXISTS idx_gold_dblp
        ON GOLD_MATCH(dblp_id)
        """,

        """
        CREATE INDEX IF NOT EXISTS idx_gold_scholar
        ON GOLD_MATCH(scholar_id)
        """
    ]

    for query in indexes:
        connection.execute(query)

    connection.commit()


def verify_indexes(connection):
    """Display all created indexes."""

    tables = [
        "DBLP_RECORD",
        "SCHOLAR_RECORD",
        "GOLD_MATCH"
    ]

    print("\n===== INDEX VERIFICATION =====")

    for table in tables:
        print(f"\nIndexes on {table}:")

        result = connection.execute(
            f"PRAGMA index_list('{table}')"
        ).fetchall()

        for index in result:
            print("-", index[1])


def main():

    if not DB_PATH.exists():
        raise FileNotFoundError(
            f"Database not found: {DB_PATH}"
        )

    connection = sqlite3.connect(DB_PATH)

    try:
        print("===== ENTITYGUARD INDEXING =====")

        create_indexes(connection)

        print("B-tree indexes created successfully!")

        verify_indexes(connection)

        print("\nINDEXING COMPLETED SUCCESSFULLY!")

    except Exception as error:
        print("Indexing failed:", error)
        raise

    finally:
        connection.close()


if __name__ == "__main__":
    main()


===== ENTITYGUARD INDEXING =====
B-tree indexes created successfully!

===== INDEX VERIFICATION =====

Indexes on DBLP_RECORD:
- idx_dblp_venue
- idx_dblp_title
- idx_dblp_year
- sqlite_autoindex_DBLP_RECORD_1

Indexes on SCHOLAR_RECORD:
- idx_scholar_venue
- idx_scholar_title
- idx_scholar_year
- sqlite_autoindex_SCHOLAR_RECORD_1

Indexes on GOLD_MATCH:
- idx_gold_scholar
- idx_gold_dblp
- sqlite_autoindex_GOLD_MATCH_1

INDEXING COMPLETED SUCCESSFULLY!


In [ ]:
import csv
import re
import sqlite3
from pathlib import Path
from collections import defaultdict, Counter

# Fixed: Jupyter/Colab environments do not have __file__, use Path.cwd() instead
ROOT = Path.cwd().resolve()

DB_PATH = ROOT / "data" / "processed" / "entityguard.db"
OUTPUT_PATH = ROOT / "outputs" / "candidate_pairs.csv"

MAX_TOKEN_DF = 40
MAX_NGRAM_DF = 100
MAX_CROSS_PAIRS_PER_BLOCK = 20000


def normalize(text):
    """Convert text into a consistent format."""
    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def tokenize(title):
    """Extract words from a title."""
    return set(normalize(title).split())


def generate_ngrams(title, n=3):
    """Generate character n-grams."""
    text = normalize(title).replace(" ", "")

    if len(text) < n:
        return set()

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


def load_records(connection, table):
    query = f"""
        SELECT record_id, title, year
        FROM {table}
    """

    return connection.execute(query).fetchall()


def add_to_blocks(blocks, key, record_id):
    blocks[key].add(record_id)


def generate_candidates(dblp_records, scholar_records):
    """Generate candidate pairs using three blocking passes."""

    blocks_dblp = defaultdict(set)
    blocks_scholar = defaultdict(set)

    # Calculate document frequency for words and n-grams.
    word_frequency = Counter()
    ngram_frequency = Counter()

    for record_id, title, year in dblp_records + scholar_records:
        word_frequency.update(tokenize(title))
        ngram_frequency.update(generate_ngrams(title))

    # PASS 1: Year + title prefix
    for records, blocks in [
        (dblp_records, blocks_dblp),
        (scholar_records, blocks_scholar)
    ]:
        for record_id, title, year in records:
            clean_title = normalize(title)

            if year is not None and len(clean_title) >= 4:
                key = f"{year}|{clean_title[:4]}"
                add_to_blocks(blocks, ("year_prefix", key), record_id)

    # PASS 2: Rare title tokens
    for records, blocks in [
        (dblp_records, blocks_dblp),
        (scholar_records, blocks_scholar)
    ]:
        for record_id, title, year in records:
            for word in tokenize(title):
                if (
                    len(word) >= 4
                    and word_frequency[word] <= MAX_TOKEN_DF
                ):
                    add_to_blocks(
                        blocks, ("rare_token", word), record_id
                    )

    # PASS 3: Rare character trigrams
    for records, blocks in [
        (dblp_records, blocks_dblp),
        (scholar_records, blocks_scholar)
    ]:
        for record_id, title, year in records:
            for gram in generate_ngrams(title):
                if ngram_frequency[gram] <= MAX_NGRAM_DF:
                    add_to_blocks(
                        blocks, ("rare_ngram", gram), record_id
                    )

    # Combine pairs from all three passes.
    candidates = defaultdict(set)
    skipped_blocks = 0

    for key, dblp_ids in blocks_dblp.items():
        scholar_ids = blocks_scholar.get(key, set())

        if not scholar_ids:
            continue

        pair_count = len(dblp_ids) * len(scholar_ids)

        if pair_count > MAX_CROSS_PAIRS_PER_BLOCK:
            skipped_blocks += 1
            continue

        for dblp_id in dblp_ids:
            for scholar_id in scholar_ids:
                candidates[(dblp_id, scholar_id)].add(key[0])

    return candidates, skipped_blocks


def evaluate_blocking(connection, candidates, total_possible):
    """Calculate reduction ratio and blocking recall."""

    gold_pairs = set(
        connection.execute(
            "SELECT dblp_id, scholar_id FROM GOLD_MATCH"
        ).fetchall()
    )

    candidate_pairs = set(candidates.keys())

    matched_gold = gold_pairs.intersection(candidate_pairs)

    recall = (
        len(matched_gold) / len(gold_pairs)
        if gold_pairs else 0
    )

    reduction = (
        1 - len(candidate_pairs) / total_possible
        if total_possible else 0
    )

    print("\n===== BLOCKING EVALUATION =====")
    print("Total possible pairs:", total_possible)
    print("Candidate pairs:", len(candidate_pairs))
    print("Gold matches:", len(gold_pairs))
    print("Gold matches found:", len(matched_gold))
    print(f"Blocking Recall: {recall:.4f}")
    print(f"Reduction Ratio: {reduction:.4f}")


def save_candidates(candidates):
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    with open(
        OUTPUT_PATH, "w", newline="", encoding="utf-8"
    ) as file:

        writer = csv.writer(file)

        writer.writerow([
            "dblp_id",
            "scholar_id",
            "blocking_rules"
        ])

        for (dblp_id, scholar_id), rules in sorted(candidates.items()):
            writer.writerow([
                dblp_id,
                scholar_id,
                ",".join(sorted(rules))
            ])


def main():
    if not DB_PATH.exists():
        raise FileNotFoundError(f"Database not found: {DB_PATH}")

    connection = sqlite3.connect(DB_PATH)

    try:
        dblp_records = load_records(connection, "DBLP_RECORD")
        scholar_records = load_records(connection, "SCHOLAR_RECORD")

        print("===== ENTITYGUARD BLOCKING =====")

        print("DBLP records:", len(dblp_records))
        print("Scholar records:", len(scholar_records))

        candidates, skipped = generate_candidates(
            dblp_records, scholar_records
        )

        total_possible = len(dblp_records) * len(scholar_records)

        save_candidates(candidates)

        print("\nCandidate pairs generated:", len(candidates))
        print("Oversized blocks skipped:", skipped)

        evaluate_blocking(
            connection, candidates, total_possible
        )

        print("\nCandidate file:", OUTPUT_PATH)
        print("BLOCKING COMPLETED SUCCESSFULLY!")

    finally:
        connection.close()


if __name__ == "__main__":
    main()


===== ENTITYGUARD BLOCKING =====
DBLP records: 2616
Scholar records: 64263

Candidate pairs generated: 234389
Oversized blocks skipped: 0

===== BLOCKING EVALUATION =====
Total possible pairs: 168112008
Candidate pairs: 234389
Gold matches: 5347
Gold matches found: 4617
Blocking Recall: 0.8635
Reduction Ratio: 0.9986

Candidate file: /content/outputs/candidate_pairs.csv
BLOCKING COMPLETED SUCCESSFULLY!


In [ ]:
import csv
import sqlite3
from pathlib import Path
from collections import defaultdict

# Project paths
# Fixed: Jupyter/Colab environments do not have __file__, use Path.cwd() instead
ROOT = Path.cwd().resolve()

DB_PATH = ROOT / "data" / "processed" / "entityguard.db"
OUTPUT_PATH = ROOT / "outputs" / "candidate_pairs.csv"


def load_records(connection, table_name):
    """Load publication records from SQLite."""

    query = f"""
        SELECT record_id, title, authors, venue, year
        FROM {table_name}
    """

    return connection.execute(query).fetchall()


def generate_candidates(dblp_records, scholar_records):
    """
    Generate candidate pairs using blocking keys.

    Records sharing at least one blocking key
    are considered possible matches.
    """

    dblp_blocks = defaultdict(set)
    scholar_blocks = defaultdict(set)

    # Build DBLP blocks
    for record in dblp_records:

        record_id, title, authors, venue, year = record

        title = (title or "").lower().strip()

        if year is not None and len(title) >= 4:
            key = ("year_prefix", year, title[:4])
            dblp_blocks[key].add(record_id)

    # Build Scholar blocks
    for record in scholar_records:

        record_id, title, authors, venue, year = record

        title = (title or "").lower().strip()

        if year is not None and len(title) >= 4:
            key = ("year_prefix", year, title[:4])
            scholar_blocks[key].add(record_id)

    # Generate candidate pairs
    candidates = defaultdict(set)

    for key, dblp_ids in dblp_blocks.items():

        scholar_ids = scholar_blocks.get(key, set())

        for dblp_id in dblp_ids:

            for scholar_id in scholar_ids:

                candidates[(dblp_id, scholar_id)].add(
                    "year_prefix"
                )

    return candidates


def remove_duplicates(candidates):
    """Ensure each DBLP-Scholar pair appears only once."""

    return {
        pair: rules
        for pair, rules in candidates.items()
    }


def save_candidates(candidates):
    """Save candidate pairs into a CSV file."""

    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        OUTPUT_PATH,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.writer(file)

        writer.writerow([
            "dblp_id",
            "scholar_id",
            "blocking_rules"
        ])

        for (dblp_id, scholar_id), rules in sorted(
            candidates.items()
        ):

            writer.writerow([
                dblp_id,
                scholar_id,
                ",".join(sorted(rules))
            ])


def main():

    if not DB_PATH.exists():
        raise FileNotFoundError(
            f"Database not found: {DB_PATH}"
        )

    connection = sqlite3.connect(DB_PATH)

    try:

        print("===== ENTITYGUARD CANDIDATE GENERATION =====")

        dblp_records = load_records(
            connection,
            "DBLP_RECORD"
        )

        scholar_records = load_records(
            connection,
            "SCHOLAR_RECORD"
        )

        print("DBLP records:", len(dblp_records))
        print("Scholar records:", len(scholar_records))

        candidates = generate_candidates(
            dblp_records,
            scholar_records
        )

        candidates = remove_duplicates(candidates)

        save_candidates(candidates)

        print("\nCandidate pairs generated:", len(candidates))

        print("Candidate file:", OUTPUT_PATH)

        print("\nCANDIDATE GENERATION COMPLETED!")

    finally:
        connection.close()


if __name__ == "__main__":
    main()

===== ENTITYGUARD CANDIDATE GENERATION =====
DBLP records: 2616
Scholar records: 64263

Candidate pairs generated: 28973
Candidate file: /content/outputs/candidate_pairs.csv

CANDIDATE GENERATION COMPLETED!


In [ ]:

import csv
from pathlib import Path
from collections import defaultdict

# Project paths
# Fixed: Jupyter/Colab environments do not have __file__, use Path.cwd() instead
ROOT = Path.cwd().resolve()

INPUT_PATH = ROOT / "outputs" / "candidate_pairs.csv"

OUTPUT_PATH = ROOT / "outputs" / "unique_candidate_pairs.csv"


def deduplicate_candidates(input_file, output_file):
    """
    Remove duplicate DBLP-Scholar candidate pairs.

    If a pair is generated by multiple blocking rules,
    retain the pair once and preserve all its rules.
    """

    unique_pairs = defaultdict(set)

    total_rows = 0

    # Read generated candidate pairs
    with open(
        input_file,
        "r",
        encoding="utf-8",
        newline=""
    ) as file:

        reader = csv.DictReader(file)

        required_columns = {
            "dblp_id",
            "scholar_id",
            "blocking_rules"
        }

        if not required_columns.issubset(
            reader.fieldnames or []
        ):
            raise ValueError(
                "Input CSV must contain dblp_id, "
                "scholar_id and blocking_rules columns."
            )

        for row in reader:

            total_rows += 1

            dblp_id = (row["dblp_id"] or "").strip()
            scholar_id = (row["scholar_id"] or "").strip()

            if not dblp_id or not scholar_id:
                continue

            pair = (dblp_id, scholar_id)

            rules = (
                row["blocking_rules"] or ""
            ).split(",")

            for rule in rules:

                rule = rule.strip()

                if rule:
                    unique_pairs[pair].add(rule)

    # Write unique candidate pairs
    output_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        output_file,
        "w",
        encoding="utf-8",
        newline=""
    ) as file:

        writer = csv.writer(file)

        writer.writerow([
            "dblp_id",
            "scholar_id",
            "blocking_rules"
        ])

        for pair, rules in sorted(unique_pairs.items()):

            writer.writerow([
                pair[0],
                pair[1],
                ",".join(sorted(rules))
            ])

    unique_count = len(unique_pairs)

    duplicate_count = total_rows - unique_count

    print("\n===== ENTITYGUARD DEDUPLICATION =====")

    print("Total input rows:", total_rows)

    print("Unique candidate pairs:", unique_count)

    print("Duplicate or invalid rows removed:", duplicate_count)

    print("Output file:", output_file)

    print("\nDEDUPLICATION COMPLETED SUCCESSFULLY!")


def main():

    if not INPUT_PATH.exists():

        raise FileNotFoundError(
            f"Candidate file not found: {INPUT_PATH}"
        )

    deduplicate_candidates(
        INPUT_PATH,
        OUTPUT_PATH
    )


if __name__ == "__main__":
    main()



===== ENTITYGUARD DEDUPLICATION =====
Total input rows: 28973
Unique candidate pairs: 28973
Duplicate or invalid rows removed: 0
Output file: /content/outputs/unique_candidate_pairs.csv

DEDUPLICATION COMPLETED SUCCESSFULLY!


In [ ]:

import csv
import random
import re
import sqlite3
import time
import tracemalloc

from pathlib import Path
from collections import defaultdict, Counter

# Project paths
# Fixed: Jupyter/Colab environments do not have __file__, use Path.cwd() instead
ROOT = Path.cwd().resolve()

DB_PATH = ROOT / "data" / "processed" / "entityguard.db"
OUTPUT_PATH = ROOT / "outputs" / "scalability_results.csv"

# Dataset percentages
DATASET_SIZES = [10, 25, 50, 75, 100]

MAX_TOKEN_DF = 40
MAX_NGRAM_DF = 100
MAX_CROSS_PAIRS_PER_BLOCK = 20000

RANDOM_SEED = 42


def normalize(text):
    """Normalize publication titles."""

    text = (text or "").lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)

    return " ".join(text.split())


def tokenize(title):
    """Extract unique title words."""

    return set(normalize(title).split())


def generate_ngrams(title, n=3):
    """Generate character trigrams."""

    text = normalize(title).replace(" ", "")

    if len(text) < n:
        return set()

    return {
        text[i:i+n]
        for i in range(len(text) - n + 1)
    }


def load_records(connection, table):
    """Load publication records."""

    query = f"""
        SELECT record_id, title, year
        FROM {table}
    """

    return connection.execute(query).fetchall()


def select_dataset(records, percentage):
    """Select a reproducible random dataset sample."""

    count = max(
        1,
        int(len(records) * percentage / 100)
    ) if records else 0

    rng = random.Random(
        RANDOM_SEED + percentage
    )

    return rng.sample(records, count)


def run_blocking(dblp_records, scholar_records):
    """Run the same three blocking passes."""

    dblp_blocks = defaultdict(set)
    scholar_blocks = defaultdict(set)

    all_records = dblp_records + scholar_records

    word_frequency = Counter()
    ngram_frequency = Counter()

    for record_id, title, year in all_records:
        word_frequency.update(tokenize(title))
        ngram_frequency.update(generate_ngrams(title))

    # Pass 1: Year + title prefix
    for records, blocks in [
        (dblp_records, dblp_blocks),
        (scholar_records, scholar_blocks)
    ]:

        for record_id, title, year in records:

            clean_title = normalize(title)

            if year is not None and len(clean_title) >= 4:

                key = (
                    "year_prefix",
                    year,
                    clean_title[:4]
                )

                blocks[key].add(record_id)

    # Pass 2: Rare title tokens
    for records, blocks in [
        (dblp_records, dblp_blocks),
        (scholar_records, scholar_blocks)
    ]:

        for record_id, title, year in records:

            for word in tokenize(title):

                if (
                    len(word) >= 4
                    and word_frequency[word] <= MAX_TOKEN_DF
                ):

                    blocks[
                        ("rare_token", word)
                    ].add(record_id)

    # Pass 3: Rare character trigrams
    for records, blocks in [
        (dblp_records, dblp_blocks),
        (scholar_records, scholar_blocks)
    ]:

        for record_id, title, year in records:

            for gram in generate_ngrams(title):

                if ngram_frequency[gram] <= MAX_NGRAM_DF:

                    blocks[
                        ("rare_ngram", gram)
                    ].add(record_id)

    # Generate unique candidate pairs
    candidates = set()
    skipped_blocks = 0

    for key, dblp_ids in dblp_blocks.items():

        scholar_ids = scholar_blocks.get(key, set())

        pair_count = len(dblp_ids) * len(scholar_ids)

        if pair_count > MAX_CROSS_PAIRS_PER_BLOCK:
            skipped_blocks += 1
            continue

        for dblp_id in dblp_ids:

            for scholar_id in scholar_ids:

                candidates.add((dblp_id, scholar_id))

    return len(candidates), skipped_blocks


def run_scalability_test():

    if not DB_PATH.exists():
        raise FileNotFoundError(
            f"Database not found: {DB_PATH}"
        )

    connection = sqlite3.connect(DB_PATH)

    results = []

    try:

        dblp_records = load_records(
            connection, "DBLP_RECORD"
        )

        scholar_records = load_records(
            connection, "SCHOLAR_RECORD"
        )

        print("===== ENTITYGUARD SCALABILITY TEST =====")

        print("Total DBLP records:", len(dblp_records))
        print("Total Scholar records:", len(scholar_records))

        for percentage in DATASET_SIZES:

            sampled_dblp = select_dataset(
                dblp_records, percentage
            )

            sampled_scholar = select_dataset(
                scholar_records, percentage
            )

            total_records = (
                len(sampled_dblp) +
                len(sampled_scholar)
            )

            total_possible = (
                len(sampled_dblp) *
                len(sampled_scholar)
            )

            # Start performance measurements
            tracemalloc.start()
            start_time = time.perf_counter()

            candidate_count, skipped = run_blocking(
                sampled_dblp,
                sampled_scholar
            )

            execution_time = (
                time.perf_counter() - start_time
            )

            current_memory, peak_memory = (
                tracemalloc.get_traced_memory()
            )

            tracemalloc.stop()

            peak_memory_mb = peak_memory / (1024 * 1024)

            reduction_ratio = (
                1 - candidate_count / total_possible
                if total_possible else 0
            )

            result = {
                "dataset_percentage": percentage,
                "dblp_records": len(sampled_dblp),
                "scholar_records": len(sampled_scholar),
                "total_records": total_records,
                "possible_pairs": total_possible,
                "candidate_pairs": candidate_count,
                "execution_time_seconds": round(
                    execution_time, 6
                ),
                "peak_memory_mb": round(
                    peak_memory_mb, 4
                ),
                "reduction_ratio": round(
                    reduction_ratio, 6
                ),
                "skipped_blocks": skipped
            }

            results.append(result)

            print(f"\nDataset size: {percentage}%")
            print("Total records:", total_records)
            print("Possible pairs:", total_possible)
            print("Candidate pairs:", candidate_count)
            print(
                "Execution time:",
                round(execution_time, 4),
                "seconds"
            )
            print(
                "Peak Python memory:",
                round(peak_memory_mb, 4),
                "MB"
            )
            print(
                "Reduction ratio:",
                round(reduction_ratio, 4)
            )
            print("Skipped blocks:", skipped)

    finally:
        connection.close()

    # Save experimental results
    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    with open(
        OUTPUT_PATH,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=list(results[0].keys())
        )

        writer.writeheader()
        writer.writerows(results)

    print("\n===== SCALABILITY TEST COMPLETED =====")
    print("Results saved at:", OUTPUT_PATH)


if __name__ == "__main__":
    run_scalability_test()


===== ENTITYGUARD SCALABILITY TEST =====
Total DBLP records: 2616
Total Scholar records: 64263

Dataset size: 10%
Total records: 6687
Possible pairs: 1677186
Candidate pairs: 119101
Execution time: 5.5456 seconds
Peak Python memory: 26.8793 MB
Reduction ratio: 0.929
Skipped blocks: 0

Dataset size: 25%
Total records: 16719
Possible pairs: 10506510
Candidate pairs: 153690
Execution time: 8.1828 seconds
Peak Python memory: 36.4471 MB
Reduction ratio: 0.9854
Skipped blocks: 0

Dataset size: 50%
Total records: 33439
Possible pairs: 42027348
Candidate pairs: 183642
Execution time: 20.56 seconds
Peak Python memory: 54.6654 MB
Reduction ratio: 0.9956
Skipped blocks: 0

Dataset size: 75%
Total records: 50159
Possible pairs: 94562514
Candidate pairs: 217017
Execution time: 25.8751 seconds
Peak Python memory: 61.2852 MB
Reduction ratio: 0.9977
Skipped blocks: 0

Dataset size: 100%
Total records: 66879
Possible pairs: 168112008
Candidate pairs: 234389
Execution time: 37.5653 seconds
Peak Python m